In [ ]:
import pandas as pd
import os
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data
data = pd.read_csv(r"")

# Basic cleaning
data = data.dropna(subset=['product', 'brand', 'description'])

# Text preprocessing
def preprocess_text(text):
    # Convert to lower case and remove punctuation
    text = text.str.lower().replace('[^\w\s]', '')
    return text

data['description'] = preprocess_text(data['description'])

# Path to the TF-IDF model and cosine similarity matrix
tfidf_model_path = r'C:\Users\Ratan\Desktop\Ecommerce prod\tfidf_vectorizer.pkl'
cosine_sim_path = r'C:\Users\Ratan\Desktop\Ecommerce prod\cosine_similarity_matrix.pkl'

<>:16: SyntaxWarning: invalid escape sequence '\w'
<>:16: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Ratan\AppData\Local\Temp\ipykernel_23908\99853437.py:16: SyntaxWarning: invalid escape sequence '\w'
  text = text.str.lower().replace('[^\w\s]', '')


In [2]:
# Check if the TF-IDF model and cosine similarity matrix already exist
if os.path.exists(tfidf_model_path) and os.path.exists(cosine_sim_path):
    # Load the TF-IDF model and cosine similarity matrix
    with open(tfidf_model_path, 'rb') as f:
        tfidf_vectorizer = pickle.load(f)
    with open(cosine_sim_path, 'rb') as f:
        cosine_sim = pickle.load(f)
    tfidf_matrix = tfidf_vectorizer.transform(data['description'])
else:
    # TF-IDF Vectorization
    tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
    tfidf_matrix = tfidf_vectorizer.fit_transform(data['description'])
    # Save the TF-IDF model for later use
    pickle.dump(tfidf_vectorizer, open(tfidf_model_path, 'wb'))
    
    # Compute cosine similarity matrix
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    # Save the cosine similarity matrix for later use
    pickle.dump(cosine_sim, open(cosine_sim_path, 'wb'))


In [4]:
def recommend_products(product_name, num_recommendations=5):
    # Check if the product exists in the dataset
    if product_name not in data['product'].values:
        return f"No products found with the name '{product_name}'. Please try another product name."

    # Get the index of the product that matches the product_name
    idx = data.index[data['product'] == product_name].tolist()[0]
    
    # Get the pairwise similarity scores of all products with that product
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort the products based on the similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the scores of the most similar products
    sim_scores = sim_scores[1:num_recommendations+1]
    
    # Get the product indices
    product_indices = [i[0] for i in sim_scores]
    
    # Return the top most similar products
    return data.iloc[product_indices][['product', 'category', 'sub_category', 'brand', 'sale_price', 'market_price', 'type', 'rating', 'description']]


In [5]:
# Example usage
product_name = input("Enter a product name to find recommendations: ")
recommended_products = recommend_products(product_name, 5)
print("Recommended Products:")
print(recommended_products)

Recommended Products:
                                   product                  category  \
16407         Authentic Chia Seeds - Black      Gourmet & World Food   
2547                            Chia Seeds      Gourmet & World Food   
25853                       Chia Mango Jam      Gourmet & World Food   
16257  Fields of Gold - Organic Chia Seeds  Foodgrains, Oil & Masala   
521                    Chia Mango Mint Jam      Gourmet & World Food   

                   sub_category        brand  sale_price  market_price  \
16407  Snacks, Dry Fruits, Nuts  Nourish You      225.00         225.0   
2547     Cooking & Baking Needs        Tiera      195.00         195.0   
25853    Sauces, Spreads & Dips      Prasukh      126.00         140.0   
16257          Masalas & Spices     PRISTINE      140.65         145.0   
521      Sauces, Spreads & Dips      Prasukh      126.00         140.0   

                           type  rating  \
16407      Roasted Seeds & Nuts     4.6   
2547          